In [ ]:
# Python libraries used in this notebook
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss, log_loss,
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score
)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Comparative Statistical Modelling and Validation of Credit Default Risk

## Logistic Regression, Probit Regression and Decision Tree Comparison

**Final Exam Project**  
**Course:** Math Concepts for Developers  
**Author:** Karamfila Stoykova  
**Date:** 30 October 2026

## Abstract

This project investigates relationships between borrower characteristics and credit default using a reproducible synthetic sample. The analysis begins with descriptive statistics and comparison of defaulted and non-defaulted borrowers. It then estimates and validates four classification models: a single-factor logistic regression, a multivariable logistic regression, a probit regression and a shallow decision tree.

The models are evaluated on an independent test sample using ROC AUC, Brier score, log loss, confusion matrices, calibration by risk groups and the difference between training and test performance. Logistic-regression coefficients are interpreted through odds ratios, while variance inflation factors are used as a multicollinearity diagnostic.

The data are fully synthetic. The project demonstrates statistical modelling and validation methods and does not represent a production model or the portfolio of any institution.

## Table of Contents

1. Introduction  
2. Synthetic Sample Generation  
3. Exploratory Statistical Analysis  
4. Statistical Modelling Theory  
5. Model Development  
6. Model Validation and Comparison  
7. Results and Model Selection  
8. Limitations  
9. Conclusion  
10. References

## 1. Introduction

### 1.1 Problem Statement

Credit default is a binary outcome: a borrower either defaults or does not default within the defined observation period. The analytical task is to estimate how borrower characteristics are associated with this outcome and to determine whether a more complex model provides a meaningful improvement over a simple benchmark.

### 1.2 Objectives

- generate a reproducible synthetic borrower sample;
- describe the distributions of the explanatory variables;
- compare defaulted and non-defaulted borrowers;
- estimate logistic and probit regression models;
- compare linear probability models with a nonlinear decision-tree benchmark;
- validate discrimination, calibration, probability accuracy and overfitting;
- interpret coefficients, odds ratios, limitations and model trade-offs.

### 1.3 Research Questions

1. Which borrower characteristics differ most clearly between defaulted and non-defaulted observations?
2. Does a multivariable logistic regression outperform a single-factor baseline?
3. Do logistic and probit links produce materially different predictions?
4. Does a shallow decision tree improve discrimination by capturing nonlinear rules?
5. Which model provides the best balance between predictive performance, calibration and interpretability?

### 1.4 Hypotheses

The full logistic regression is expected to outperform the DTI-only baseline because it includes several distinct dimensions of borrower risk. Probit regression is expected to perform similarly to logistic regression because the logistic and normal cumulative distribution functions have similar shapes. The decision tree may capture nonlinear relationships, but its probability estimates may be less stable or less well calibrated.

## 2. Synthetic Sample Generation

### 2.1 Variables

| Variable | Description |
|---|---|
| `age` | Borrower age in years |
| `monthly_income` | Monthly disposable income |
| `loan_amount` | Outstanding loan amount |
| `dti` | Debt-to-income ratio |
| `credit_utilisation` | Utilised share of available revolving credit |
| `delinquencies_12m` | Number of recent delinquency events |
| `default_flag` | Binary outcome: 1 default, 0 non-default |

### 2.2 Data-Generating Process

The binary outcome is generated from a logistic probability:

$$p_i=P(Y_i=1\mid X_i)=\frac{1}{1+e^{-z_i}}$$

where the latent risk score contains linear effects, a DTI-utilisation interaction and a mild nonlinear loan-burden effect:

$$z_i=\beta_0+\beta_1DTI_i+\beta_2U_i+\beta_3L_i-\beta_4Income_i+\beta_5Age_i+\beta_6(DTI_iU_i)+\beta_7LoanBurden_i^2$$

The interaction makes the effect of DTI stronger at high utilisation. The nonlinear term gives the decision tree a realistic opportunity to detect structure not represented in the basic linear models.

In [ ]:
RANDOM_SEED = 42
N = 5000
rng = np.random.default_rng(RANDOM_SEED)

age = np.clip(np.rint(rng.normal(42, 11, N)), 18, 75).astype(int)
monthly_income = np.clip(rng.lognormal(np.log(2600), 0.42, N), 800, 12000)
loan_amount = np.clip(rng.lognormal(np.log(18000), 0.65, N), 1000, 150000)
dti = rng.beta(2.4, 4.2, N)
credit_utilisation = rng.beta(2.0, 2.7, N)

delinquency_p = np.clip(0.03 + 0.16*dti + 0.14*credit_utilisation, 0, 0.50)
delinquencies_12m = rng.binomial(3, delinquency_p, N)

income_z = (np.log(monthly_income) - np.log(monthly_income).mean()) / np.log(monthly_income).std()
age_z = (age - age.mean()) / age.std()
loan_burden = loan_amount / (12 * monthly_income)

z = (
    -4.10 + 2.20*dti + 1.45*credit_utilisation
    + 0.70*delinquencies_12m - 0.35*income_z
    + 0.10*age_z + 1.35*(dti*credit_utilisation)
    + 0.18*np.minimum(loan_burden, 4)**2
)
true_pd = 1/(1+np.exp(-z))
default_flag = rng.binomial(1, true_pd, N)

data = pd.DataFrame({
    'age': age,
    'monthly_income': monthly_income,
    'loan_amount': loan_amount,
    'dti': dti,
    'credit_utilisation': credit_utilisation,
    'delinquencies_12m': delinquencies_12m,
    'default_flag': default_flag
})

print('Sample shape:', data.shape)
print(f'Default rate: {data.default_flag.mean():.2%}')
data.head()

In [ ]:
# Data-quality checks
checks = pd.Series({
    'No missing values': not data.isna().any().any(),
    'No duplicate rows': not data.duplicated().any(),
    'Age in expected range': data.age.between(18,75).all(),
    'Positive income': (data.monthly_income > 0).all(),
    'Positive loan amount': (data.loan_amount > 0).all(),
    'DTI in [0,1]': data.dti.between(0,1).all(),
    'Utilisation in [0,1]': data.credit_utilisation.between(0,1).all(),
    'Binary target': data.default_flag.isin([0,1]).all()
})
checks

## 3. Exploratory Statistical Analysis

The exploratory analysis is deliberately focused on the modelling question. It describes the sample, compares defaulted and non-defaulted borrowers and checks linear relationships between candidate predictors.

In [ ]:
# Overall descriptive statistics
features = ['age','monthly_income','loan_amount','dti','credit_utilisation','delinquencies_12m']
data[features].describe().T.round(3)

In [ ]:
# Comparison by observed outcome
group_comparison = data.groupby('default_flag')[features].agg(['mean','median','std']).T
group_comparison.round(3)

In [ ]:
# Compact distribution plots
plot_vars = ['monthly_income','loan_amount','dti','credit_utilisation','delinquencies_12m']
fig, axes = plt.subplots(2,3,figsize=(14,8))
axes=axes.flatten()
for ax, var in zip(axes, plot_vars):
    ax.hist(data.loc[data.default_flag==0,var], bins=30, alpha=0.55, density=True, label='Non-default')
    ax.hist(data.loc[data.default_flag==1,var], bins=30, alpha=0.55, density=True, label='Default')
    ax.set_title(var)
    ax.set_ylabel('Density')
    ax.legend()
for ax in axes[len(plot_vars):]: ax.set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation matrix for candidate predictors and the target
corr = data[features + ['default_flag']].corr()
fig, ax = plt.subplots(figsize=(9,7))
im=ax.imshow(corr,cmap='coolwarm',vmin=-1,vmax=1)
ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns,rotation=45,ha='right'); ax.set_yticklabels(corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center',fontsize=8)
plt.colorbar(im,ax=ax,label='Pearson correlation')
plt.title('Correlation Matrix'); plt.tight_layout(); plt.show()

## 4. Statistical Modelling Theory

### 4.1 Logistic Regression

Logistic regression models the log-odds of default:

$$\log\left(\frac{p_i}{1-p_i}\right)=\beta_0+\beta_1x_{1i}+\cdots+\beta_kx_{ki}$$

The odds ratio associated with a one-unit increase in $x_j$ is:

$$OR_j=e^{\beta_j}$$

For variables measured on a 0-to-1 scale, an increase of 0.10 is often more interpretable:

$$OR_{0.10}=e^{0.10\beta_j}$$

### 4.2 Probit Regression

Probit regression uses the cumulative standard normal distribution:

$$P(Y_i=1\mid X_i)=\Phi(X_i\beta)$$

Logit and probit coefficients are not directly comparable because the link functions use different scales.

### 4.3 Decision Tree

A decision tree partitions the feature space through sequential rules. A shallow tree can detect nonlinearities and interactions while remaining understandable. The tree is constrained to reduce overfitting.

### 4.4 Validation Measures

Discrimination is evaluated with ROC AUC. Probability accuracy is evaluated with Brier score and log loss:

$$Brier=\frac{1}{n}\sum_{i=1}^n(p_i-y_i)^2$$

$$LogLoss=-\frac{1}{n}\sum_{i=1}^n[y_i\log(p_i)+(1-y_i)\log(1-p_i)]$$

## 5. Model Development

The sample is divided into 70% training and 30% test observations using stratification. All model comparison is based on the untouched test sample.

In [ ]:
X = data[features].copy()
y = data['default_flag'].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED
)
print('Training observations:', len(X_train), f'| default rate {y_train.mean():.2%}')
print('Test observations:', len(X_test), f'| default rate {y_test.mean():.2%}')

In [ ]:
# Model 1: DTI-only baseline logistic regression
baseline_logit = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_SEED))
])
baseline_logit.fit(X_train[['dti']], y_train)

# Model 2: full multivariable logistic regression
full_logit = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_SEED))
])
full_logit.fit(X_train, y_train)

# Model 3: probit regression with standardised features
scaler_probit = StandardScaler()
X_train_s = pd.DataFrame(scaler_probit.fit_transform(X_train), columns=features, index=X_train.index)
X_test_s = pd.DataFrame(scaler_probit.transform(X_test), columns=features, index=X_test.index)
probit = sm.Probit(y_train, sm.add_constant(X_train_s)).fit(disp=False)

# Model 4: shallow decision tree
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=80, random_state=RANDOM_SEED)
tree.fit(X_train, y_train)

print('Models fitted successfully.')

### 5.1 Logistic Coefficients and Odds Ratios

The coefficient table uses standardised predictors, so each coefficient represents an increase of one training-sample standard deviation. This makes relative coefficient magnitudes easier to compare.

In [ ]:
coef = full_logit.named_steps['model'].coef_[0]
coefficient_table = pd.DataFrame({
    'Variable': features,
    'Standardised coefficient': coef,
    'Odds ratio per 1 SD': np.exp(coef)
}).sort_values('Standardised coefficient', key=np.abs, ascending=False)
coefficient_table.round(4)

In [ ]:
# Multicollinearity diagnostic on standardised training predictors
vif_table = pd.DataFrame({
    'Variable': features,
    'VIF': [variance_inflation_factor(X_train_s.values, i) for i in range(len(features))]
})
vif_table.round(3)

## 6. Model Validation and Comparison

Validation covers independent test performance, discrimination, probability accuracy, classification performance, calibration, overfitting and multicollinearity.

In [ ]:
models = {
    'Baseline Logit': {
        'train': baseline_logit.predict_proba(X_train[['dti']])[:,1],
        'test': baseline_logit.predict_proba(X_test[['dti']])[:,1]
    },
    'Full Logit': {
        'train': full_logit.predict_proba(X_train)[:,1],
        'test': full_logit.predict_proba(X_test)[:,1]
    },
    'Probit': {
        'train': np.asarray(probit.predict(sm.add_constant(X_train_s))),
        'test': np.asarray(probit.predict(sm.add_constant(X_test_s)))
    },
    'Decision Tree': {
        'train': tree.predict_proba(X_train)[:,1],
        'test': tree.predict_proba(X_test)[:,1]
    }
}

rows=[]
threshold=0.30
for name,p in models.items():
    pred=(p['test']>=threshold).astype(int)
    rows.append({
        'Model':name,
        'Train AUC':roc_auc_score(y_train,p['train']),
        'Test AUC':roc_auc_score(y_test,p['test']),
        'AUC gap':roc_auc_score(y_train,p['train'])-roc_auc_score(y_test,p['test']),
        'Brier score':brier_score_loss(y_test,p['test']),
        'Log loss':log_loss(y_test,p['test']),
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred,zero_division=0),
        'Recall':recall_score(y_test,pred,zero_division=0),
        'F1':f1_score(y_test,pred,zero_division=0)
    })
results=pd.DataFrame(rows).set_index('Model').sort_values('Test AUC',ascending=False)
results.round(4)

In [ ]:
# ROC curves on the test sample
plt.figure(figsize=(8,6))
for name,p in models.items():
    fpr,tpr,_=roc_curve(y_test,p['test'])
    auc=roc_auc_score(y_test,p['test'])
    plt.plot(fpr,tpr,label=f'{name}: {auc:.3f}')
plt.plot([0,1],[0,1],'k--',label='Random ranking')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves on the Test Sample'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Calibration by test-sample deciles
def calibration_table(probabilities, observed, groups=10):
    frame=pd.DataFrame({'predicted':probabilities,'observed':np.asarray(observed)})
    frame['group']=pd.qcut(frame['predicted'],q=groups,duplicates='drop')
    return frame.groupby('group',observed=False).agg(
        borrowers=('observed','size'),
        average_predicted=('predicted','mean'),
        observed_rate=('observed','mean')
    ).reset_index(drop=True)

calibration_tables={name:calibration_table(p['test'],y_test) for name,p in models.items()}

fig,axes=plt.subplots(2,2,figsize=(11,9))
for ax,(name,tab) in zip(axes.flatten(),calibration_tables.items()):
    ax.plot(tab.average_predicted,tab.observed_rate,'o-',label=name)
    ax.plot([0,tab.average_predicted.max()*1.1],[0,tab.average_predicted.max()*1.1],'k--')
    ax.set_title(name); ax.set_xlabel('Average predicted probability'); ax.set_ylabel('Observed default rate')
    ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrices at threshold 30%
fig,axes=plt.subplots(2,2,figsize=(9,8))
for ax,(name,p) in zip(axes.flatten(),models.items()):
    cm=confusion_matrix(y_test,(p['test']>=0.30).astype(int))
    ax.imshow(cm,cmap='Blues')
    for i in range(2):
        for j in range(2): ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=12)
    ax.set_title(name); ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xlabel('Predicted class'); ax.set_ylabel('Observed class')
plt.tight_layout(); plt.show()

In [ ]:
# Threshold sensitivity for the preferred interpretable model
thresholds=[0.20,0.30,0.40,0.50]
p=models['Full Logit']['test']
threshold_rows=[]
for t in thresholds:
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,pred).ravel()
    threshold_rows.append({
        'Threshold':t,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred,zero_division=0),
        'Recall':recall_score(y_test,pred,zero_division=0),
        'Specificity':tn/(tn+fp),
        'F1':f1_score(y_test,pred,zero_division=0)
    })
pd.DataFrame(threshold_rows).round(4)

In [ ]:
# Visualise the shallow decision tree
plt.figure(figsize=(18,8))
plot_tree(tree,feature_names=features,class_names=['Non-default','Default'],filled=True,rounded=True,fontsize=8)
plt.title('Shallow Decision Tree Benchmark')
plt.show()

## 7. Results and Model Selection

The final model decision should not be based on one statistic. The preferred model should combine acceptable test-sample discrimination, low Brier score and log loss, reasonable calibration, a small train-test AUC gap and clear interpretation.

The DTI-only model provides a transparent benchmark but omits important risk information. The full logistic model incorporates several borrower characteristics and supports odds-ratio interpretation. Probit provides a robustness check for the link-function assumption. The shallow decision tree tests whether simple nonlinear partitions materially improve performance.

Run all cells and use the final comparison table and calibration plots to state the preferred model. If logistic and probit performance is nearly identical, logistic regression is generally easier to communicate through odds ratios. A tree should only be preferred if its incremental performance is material and its calibration and train-test stability remain acceptable.

## 8. Limitations

1. The sample is synthetic and embeds assumed relationships.
2. The outcome is generated from a known mathematical process, so results cannot be generalised to real borrowers.
3. The candidate variables exclude macroeconomic, behavioural time-series and product information.
4. A single train-test split introduces sampling variability. Cross-validation or repeated splits would provide a broader stability assessment.
5. Calibration is assessed descriptively by groups rather than through a complete validation framework.
6. Threshold selection does not incorporate the economic cost of false positives and false negatives.
7. The project does not address reject inference, sample selection, missing-data mechanisms or temporal drift.

## 9. Conclusion

This project follows one coherent analytical path: generate a borrower sample, explore relationships, estimate competing models and validate them on independent observations.

The comparison demonstrates why a single accuracy measure is insufficient. ROC AUC evaluates ranking, Brier score and log loss evaluate probability accuracy, calibration groups compare predicted and observed risk, and the train-test gap provides an overfitting diagnostic. Logistic coefficients and odds ratios support interpretation, while probit and the shallow decision tree provide alternative modelling assumptions.

The conclusions apply only to the synthetic sample. The value of the project lies in the modelling and validation process rather than in the numerical results themselves.

## 10. References

1. Course lecture: *Statistics*.
2. Course lecture: *Linear Algebra*.
3. Course lecture: *Calculus*.
4. Course document: *Final Exam Project Guidelines*.
5. scikit-learn documentation: LogisticRegression, DecisionTreeClassifier and model evaluation metrics.
6. statsmodels documentation: Probit and variance inflation factor.
7. NumPy, pandas and Matplotlib documentation.

